# Active case finding in Vietnam: results

Loads a stored run of the analysis and summarises it: fit to the calibration targets,
epidemiological impact, costs and DALYs, and cost-effectiveness, each as a median with an
interval across posterior draws. Also writes the CSV files to share with collaborators.

| Scenario | Screening |
| --- | --- |
| `baseline` | None; routine services only |
| `acf_xpert` | Xpert as the front-line test |
| `acf_cxr_xpert` | Chest X-ray front-line, confirmatory Xpert for X-ray positives |

The second algorithm tests in series, so a case must be X-ray positive **and** Xpert
positive to be found. It is therefore less sensitive than front-line Xpert and cheaper per
person screened — the trade-off this analysis quantifies.

Everything the analysis simulates is defined in `vacf.economics.study`. Change it there,
not here.

Runs come from [10-ACF-CEA-Run](10-ACF-CEA-Run.ipynb) locally or from
`scripts/massive/calibrate.py` on MASSIVE; both write the same layout under
`outputs/<run>/`. Nothing here re-runs the model.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from vacf.calibration import observed_targets
from vacf.constants import OUTPUT_PATH
from vacf.economics import analysis, study

pd.set_option("display.width", 200)

print(study.describe())

banner = study.provisional_warning()
if banner:
    warnings.warn(banner, stacklevel=1)
    print("\n*** " + banner + " ***")

## Load a run

Set `RUN` to a run directory. `outputs/local-sample-n100` is a short local run that has not
converged, for development only; for reporting, point at a downloaded MASSIVE job
(`outputs/<job_id>`).

Intervals are the central 95% of draws. Differences between arms are taken within each draw,
so uncertainty they share (the calibrated epidemic) does not widen the interval of the
difference.

In [ ]:
RUN = OUTPUT_PATH / "local-sample-n100"
posterior = analysis.load_posterior_outputs(RUN)

worst_rhat = posterior["posterior_summary"]["r_hat"].max()
print(f"{RUN.name}: {len(posterior['draws'])} posterior draws, worst r_hat {worst_rhat:.3f}")
if worst_rhat > 1.01:
    print("NOT CONVERGED. Treat these results as a pipeline demonstration only.")
posterior["posterior_summary"][["mean", "sd", "ess_bulk", "r_hat"]]

### Screening sensitivity

Values follow the Kiribati methods appendix. The full algorithm (symptom screening, chest
X-ray and Xpert) *defines* the model's active TB, so it has sensitivity one in every state
by construction; the X-ray approach is scored by asking whether it would also have found
each person the full algorithm found. Symptomatic disease is detected with certainty
because symptom screening is the first step and defines symptom status in the model.

Because the confirmatory Xpert step has sensitivity one, the serial product for scenario 2
reduces exactly to the sensitivity of chest X-ray alone.

In [ ]:
pd.DataFrame(study.sensitivity_table()).T[list(study.DISEASE_STATES)]

## Fit to calibration targets

Median and 95% interval of each target quantity across draws, against the data in the units
the likelihood compares.

In [ ]:
observed = observed_targets()
fit = analysis.with_prevalence_decline(
    posterior["calibration_fit"], base_time=observed["decline"].index[0]
)
fit = fit[fit["year"].between(1990, 2025)]

# (column, title, y-axis label, display scale)
FIT_PANELS = [
    ("notifications", "Notifications", "Per year", 1.0),
    ("deaths", "TB deaths", "Per year", 1.0),
    ("latent", "Ever infected, adults", "%", 100.0),
    ("prevalence", "Pulmonary TB prevalence, adults", "Per 100,000", 1e5),
    ("decline", "Prevalence relative to first survey", "Ratio", 1.0),
    ("infectiousness", "High-infectious share of prevalent TB", "%", 100.0),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for ax, (name, title, ylabel, scale) in zip(axes.flat, FIT_PANELS):
    bands = analysis.quantile_bands(fit, name, ["year"]) * scale
    ax.fill_between(bands.index, bands["lower"], bands["upper"], color="tab:red", alpha=0.25,
                    label="95% interval")
    ax.plot(bands.index, bands["median"], color="tab:red", label="Median")
    target = observed[name].loc[1990:2025]
    ax.scatter(target.index, target * scale, color="black", zorder=3, label="Target")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("Year")
axes[0, 0].legend(frameon=False)
fig.suptitle("Fit to calibration targets across posterior draws")
fig.tight_layout()
plt.show()

## Epidemiological impact

Cumulative incidence and deaths over the study horizon, and the share averted relative to no
active case finding.

In [ ]:
analysis.interval_summary(
    posterior["epi_summary"],
    ["cum_incidence", "cum_deaths", "cum_incidence_averted_pct", "cum_deaths_averted_pct"],
).round(1)

In [ ]:
start, end = study.ACF_START - 5, study.HORIZON[1]
recent = posterior["epi"][posterior["epi"]["year"].between(start, end)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (column, title) in zip(axes, [("incidence", "TB incidence"), ("deaths", "TB deaths")]):
    bands = analysis.quantile_bands(recent, column, ["arm", "year"])
    for arm in study.SCENARIOS:
        band = bands.loc[arm.name]
        (line,) = ax.plot(band.index, band["median"], label=arm.label)
        ax.fill_between(band.index, band["lower"], band["upper"], color=line.get_color(), alpha=0.2)
    ax.axvspan(study.ACF_START, study.ACF_START + study.ACF_DURATION, color="grey", alpha=0.12)
    ax.set_title(title)
    ax.set_ylabel("Per year")
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False)
fig.suptitle("Median and 95% interval across draws; grey marks the campaign")
fig.tight_layout()
plt.show()

## Costs and DALYs

Discounted totals over the study horizon: people screened in millions, costs in US$
millions, DALYs in thousands.

In [ ]:
econ_draws = posterior["economic_summary"].assign(
    screened_millions=lambda t: t["n_screened"] / 1e6,
    cost_musd=lambda t: t["cost"] / 1e6,
    dalys_thousands=lambda t: t["dalys"] / 1e3,
)
analysis.interval_summary(
    econ_draws, ["screened_millions", "cost_musd", "dalys_thousands"]
).round(1)

In [ ]:
medians = posterior["economic_summary"].groupby("label", sort=False).median(numeric_only=True)
costs = medians[["cost_provider", "cost_patient"]].rename(
    columns={"cost_provider": "Provider", "cost_patient": "Patient"}
)
dalys = medians[["ylls", "ylds", "post_tb_dalys"]].rename(
    columns={"ylls": "YLL", "ylds": "YLD", "post_tb_dalys": "Post-TB"}
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
(costs / 1e6).plot.barh(stacked=True, ax=axes[0])
axes[0].set_xlabel("US$ millions, discounted")
axes[0].set_title("Cost by perspective")
(dalys / 1e3).plot.barh(stacked=True, ax=axes[1])
axes[1].set_xlabel("Thousands, discounted")
axes[1].set_title("DALYs by component")
axes[0].invert_yaxis()
axes[0].set_ylabel("")
fig.suptitle("Median of each component across draws")
fig.tight_layout()
plt.show()

## Cost-effectiveness

The first table pairs each campaign with no active case finding within every draw. The
efficiency frontier and ICERs are then computed on costs and DALYs averaged over draws, the
usual summary of a probabilistic analysis: per-draw ICERs are undefined in draws where an
option is dominated, and ratios of noisy differences are heavy-tailed. An option is
**dominated** when another is both cheaper and averts more DALYs, and **extended-dominated**
when a combination of other options on the frontier does better; each ICER is against the
next cheaper option on the frontier. The count table shows how often each option is on the
frontier across draws.

In [ ]:
paired = analysis.incremental_vs_baseline(posterior["economic_summary"]).assign(
    inc_cost_musd=lambda t: t["inc_cost"] / 1e6,
    dalys_averted_thousands=lambda t: t["dalys_averted"] / 1e3,
)
analysis.interval_summary(paired, ["inc_cost_musd", "dalys_averted_thousands"]).round(1)

In [ ]:
expected = analysis.expected_incremental(posterior["economic_summary"])
display(expected[["cost", "dalys", "status", "inc_cost", "inc_dalys_averted", "icer"]])

labels = [arm.label for arm in study.SCENARIOS]
pd.crosstab(posterior["incremental"]["label"], posterior["incremental"]["status"]).reindex(labels)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for label, group in paired.groupby("label", sort=False):
    ax.scatter(group["dalys_averted_thousands"], group["inc_cost_musd"], s=14, alpha=0.6,
               label=label)
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("DALYs averted vs baseline (thousands, discounted)")
ax.set_ylabel("Incremental cost vs baseline (US$ millions, discounted)")
ax.set_title("Cost-effectiveness plane: one point per posterior draw")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## CSV files to share

Median, interquartile range and 95% interval of the epidemiological and economic outputs,
with a unit for every quantity, in `<run>/csv/` alongside a README that defines them. Runs
written by the current scripts already include these; this cell (re)writes them from the
stored tables, for runs made before they were added.

In [ ]:
paths = analysis.write_summary_csvs(posterior, RUN / "csv")
for path in paths:
    print(path.relative_to(OUTPUT_PATH))
pd.read_csv(RUN / "csv" / "epi_outcomes.csv").head(6)

## Caveats

1. **Sensitivities are carried over from Kiribati.** They assume Vietnamese active TB is
   delimited the same way — by what the full algorithm detects — and that X-ray performance
   transfers. The two asymptomatic X-ray values, 0.2 and 0.9, lie within the uniform
   priors estimated there, 𝒰(0.07, 0.52) and 𝒰(0.72, 0.93). They cannot be identified from
   Vietnamese data, since screening is off during calibration, so vary them over those
   ranges in one-way sensitivity analysis (`study.CXR_SENSITIVITY_RANGE`).
2. **Check convergence.** The worst r_hat is printed when a run is loaded. Do not report
   results from a run above 1.01.
3. **Intervals reflect calibration uncertainty only.** Screening sensitivities, costs and
   disability weights are fixed at their point values; vary them in sensitivity analysis.
4. **Homogeneous mixing** is assumed, so no contact survey data informs the age pattern of
   transmission, and the upstream posterior does not transfer.
5. **Coverage is annual.** 80% per year over ten years means roughly ten screening rounds,
   so total screening volume is about ten times the eligible population and screening cost
   scales with campaign duration.